In [ ]:
from pathlib import Path
import pandas as pd

raw_path = Path("../raw")

files = [
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "product_category_name_translation.csv",
]

summary = []

for file_name in files:
    file_path = raw_path / file_name
    df = pd.read_csv(file_path)
    summary.append(
        {
            "file_name": file_name,
            "num_rows": df.shape[0],
            "num_columns": df.shape[1],
        }
    )
pd.DataFrame(summary)

customers = pd.read_csv(raw_path / "olist_customers_dataset.csv")
orders = pd.read_csv(raw_path / "olist_orders_dataset.csv")
order_items = pd.read_csv(raw_path / "olist_order_items_dataset.csv")
payments = pd.read_csv(raw_path / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(raw_path / "olist_order_reviews_dataset.csv")
products = pd.read_csv(raw_path / "olist_products_dataset.csv")
category_translation = pd.read_csv(raw_path / "product_category_name_translation.csv")

tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "category_translation": category_translation,
}

for name, df in tables.items():
    print(f"\n{name.upper()}")
    print(list(df.columns))


# Convert order dates to datetime
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

# Keep completed transactions only
delivered_orders = orders.loc[orders["order_status"] == "delivered"].copy()

# Aggregate item-level data to one row per order
items_by_order = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        product_revenue=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
)

# Aggregate payments and reviews to one row per order
payments_by_order = (
    payments
    .groupby("order_id", as_index=False)
    .agg(payment_value=("payment_value", "sum"))
)

reviews_by_order = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(review_score=("review_score", "mean"))
)

# Create the clean order-level fact table
order_fact = (
    delivered_orders
    .merge(customers, on="customer_id", how="left", validate="many_to_one")
    .merge(items_by_order, on="order_id", how="left", validate="one_to_one")
    .merge(payments_by_order, on="order_id", how="left", validate="one_to_one")
    .merge(reviews_by_order, on="order_id", how="left", validate="one_to_one")
)

# Create delivery-performance fields
order_fact["order_value"] = (
    order_fact["product_revenue"] + order_fact["freight_value"]
)

order_fact["delivery_days"] = (
    order_fact["order_delivered_customer_date"]
    - order_fact["order_purchase_timestamp"]
).dt.total_seconds() / 86400

order_fact["is_late"] = (
    order_fact["order_delivered_customer_date"]
    > order_fact["order_estimated_delivery_date"]
)

# Check the finished table
order_fact[
    [
        "order_id",
        "customer_unique_id",
        "order_value",
        "payment_value",
        "review_score",
        "delivery_days",
        "is_late",
    ]
].head()

print("Delivered orders:", len(order_fact))
print("Total order value:", round(order_fact["order_value"].sum(), 2))
print("Missing order value:", order_fact["order_value"].isna().sum())
print("Missing review score:", order_fact["review_score"].isna().sum()) 


CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

PRODUCTS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

CATEGORY_TRANSLATION
['product_category_name', 'product_category_name_english']
Delivered orders: 96478
Tota